## Consensus trees versus supertrees evaluations
heatmaps

In [4]:
#!/usr/bin/env python3
"""
Consensus trees vs. MR(+) supertrees evaluation + heatmaps
"""

import os
import re
import glob
import math
import statistics
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dendropy


# Config

DATA_ROOT = "."
OUTDIR = "eval_consensus_vs_supertrees"
os.makedirs(OUTDIR, exist_ok=True)

USE_CACHE_IF_AVAILABLE = True
FORCE_RECOMPUTE = False

EPS = 1e-8
USE_MEDIAN = False
DROP_INTERNAL_LABELS = True

group_configs = {
    "Amphibians": {
        "folder": "Amphibians_completed_sets",
        "base_file": os.path.join("base_trees", "amphibians_base_trees20.txt"),
        "prefix": "amphibians",
    },
    "Squamates": {
        "folder": "Squamates_completed_sets",
        "base_file": os.path.join("base_trees", "squamates_base_trees20.txt"),
        "prefix": "squamates",
    },
    "Mammals": {
        "folder": "Mammals_completed_sets",
        "base_file": os.path.join("base_trees", "mammals_base_trees20.txt"),
        "prefix": "mammals",
    },
    "Sharks": {
        "folder": "Sharks_completed_sets",
        "base_file": os.path.join("base_trees", "sharks_base_trees20.txt"),
        "prefix": "sharks",
    },
}

methods = {
    "original": ("original", "Proposed"),
    "at_root": ("root_attach", "Root Attach"),
    "nearest_parent": ("nearest_parent", "Nearest Parent"),
    "no_mcs": ("no_mcs", "No MCS"),
    "supertree": ("supertree", "Supertree-based"),
}

# Fixed plot order
group_order = ["Amphibians", "Mammals", "Sharks", "Squamates"]
method_order = ["original", "at_root", "nearest_parent", "no_mcs", "supertree"]

METRICS_PKL = os.path.join(OUTDIR, "consensus_vs_supertree_metrics.pkl")
METRICS_CSV = os.path.join(OUTDIR, "consensus_vs_supertree_metrics.csv")

HEAT_RF_PNG = os.path.join(OUTDIR, "heatmap_consensus_vs_supertree_rootedRF.png")
HEAT_RF_SVG = os.path.join(OUTDIR, "heatmap_consensus_vs_supertree_rootedRF.svg")
HEAT_BSD_PNG = os.path.join(OUTDIR, "heatmap_consensus_vs_supertree_BSD.png")
HEAT_BSD_SVG = os.path.join(OUTDIR, "heatmap_consensus_vs_supertree_BSD.svg")

HEAT_COMBINED_PNG = os.path.join(OUTDIR, "heatmap_consensus_vs_supertree_combined.png")
HEAT_COMBINED_SVG = os.path.join(OUTDIR, "heatmap_consensus_vs_supertree_combined.svg")

CMAP = "viridis"
DPI = 240


# IO helpers

def load_newick_records(path: str):
    txt = Path(path).read_text(encoding="utf-8")
    parts = [p.strip() for p in re.split(r";\s*", txt) if p.strip()]
    return [p + ";" for p in parts]

def extract_subset_index(path: str) -> int:
    base = os.path.basename(path)
    nums = re.findall(r"(\d+)", base)
    if not nums:
        raise ValueError(f"Could not find subset index in filename: {base}")
    return int(nums[-1])

def ensure_semicolon(nwk: str) -> str:
    nwk = nwk.strip()
    return nwk if nwk.endswith(";") else (nwk + ";")


# Branch lengths by split-wise averaging

def taxon_union_namespace(super_nwk, source_nwks):
    tns = dendropy.TaxonNamespace()
    _ = dendropy.Tree.get(data=super_nwk, schema="newick", taxon_namespace=tns)
    for s in source_nwks:
        _ = dendropy.Tree.get(data=s, schema="newick", taxon_namespace=tns)
    return tns

def tree_edge_cluster_map(t: dendropy.Tree):
    t.encode_bipartitions()
    present = {lf.taxon.label for lf in t.leaf_node_iter()}

    edge_len_by_cluster = {}
    for e in t.postorder_edge_iter():
        if e.head_node is t.seed_node:
            continue
        mask = e.bipartition.leafset_bitmask
        cluster = set()
        for idx, taxon in enumerate(t.taxon_namespace):
            if mask & (1 << idx):
                cluster.add(taxon.label)
        cluster = frozenset(cluster)
        comp = frozenset(present - cluster)
        elen = float(e.length) if (e.length is not None) else 0.0
        edge_len_by_cluster[cluster] = elen
        edge_len_by_cluster[comp] = elen

    leaf_pendant = {}
    for lf in t.leaf_node_iter():
        elen = float(lf.edge.length) if (lf.edge.length is not None) else 0.0
        leaf_pendant[lf.taxon.label] = elen

    return present, edge_len_by_cluster, leaf_pendant

def set_branch_lengths_by_split_average(super_nwk, source_nwks, drop_internal_labels=True):
    tns = taxon_union_namespace(super_nwk, source_nwks)
    S = dendropy.Tree.get(data=super_nwk, schema="newick", taxon_namespace=tns)
    S.is_rooted = True
    S.encode_bipartitions()

    src_infos = []
    for s in source_nwks:
        t = dendropy.Tree.get(data=s, schema="newick", taxon_namespace=tns)
        t.is_rooted = True
        present, edge_map, leaf_pendant = tree_edge_cluster_map(t)
        src_infos.append((present, edge_map, leaf_pendant))

    union_taxa = {lf.taxon.label for lf in S.leaf_node_iter()}

    for e in S.postorder_edge_iter():
        if e.head_node is S.seed_node or e.head_node.is_leaf():
            continue
        mask = e.bipartition.leafset_bitmask
        sideA = {taxon.label for idx, taxon in enumerate(S.taxon_namespace) if mask & (1 << idx)}
        sideB = union_taxa - sideA
        cluster_small = frozenset(sideA if len(sideA) <= len(sideB) else sideB)

        donors = []
        for present, edge_map, _leafpend in src_infos:
            if cluster_small.issubset(present) and len(present - cluster_small) >= 1:
                if cluster_small in edge_map:
                    donors.append(edge_map[cluster_small])

        e.length = float(statistics.median(donors) if USE_MEDIAN else (sum(donors) / len(donors))) if donors else EPS

    for lf in S.leaf_node_iter():
        label = lf.taxon.label
        vals = [leafpend[label] for _present, _emap, leafpend in src_infos if label in leafpend]
        lf.edge.length = float(statistics.median(vals) if USE_MEDIAN else (sum(vals) / len(vals))) if vals else EPS

    if drop_internal_labels:
        for nd in S.internal_nodes():
            nd.label = None
        return S.as_string(
            schema="newick",
            suppress_rooting=False,
            suppress_internal_node_labels=True
        ).strip()

    return S.as_string(schema="newick", suppress_rooting=False).strip()


# Majority rule consensus

def build_majority_consensus_newick(source_nwks, leaf_labels, min_freq=0.5):
    tns = dendropy.TaxonNamespace(list(leaf_labels))
    treelist = dendropy.TreeList(taxon_namespace=tns)

    for s in source_nwks:
        t = dendropy.Tree.get(data=s, schema="newick", taxon_namespace=tns)
        t.is_rooted = True
        t.retain_taxa_with_labels(leaf_labels)
        treelist.append(t)

    try:
        cons = treelist.consensus(min_freq=min_freq)
    except Exception:
        try:
            from dendropy.calculate import treesum
            cons = treesum.consensus_tree(treelist, min_freq=min_freq)
        except Exception as e:
            raise RuntimeError(f"Could not construct consensus tree with dendropy: {e}")

    cons.is_rooted = True
    topo_nwk = cons.as_string(
        schema="newick",
        suppress_rooting=False,
        suppress_internal_node_labels=True
    ).strip()
    return ensure_semicolon(topo_nwk)


# Metrics: normalized rooted RF + normalized BSD

def rooted_nontrivial_clades(tree: dendropy.Tree):
    labels = [lf.taxon.label for lf in tree.leaf_node_iter()]
    leafset = set(labels)
    n = len(leafset)
    clades = set()
    if n < 4:
        return leafset, clades

    for nd in tree.postorder_node_iter():
        if nd is tree.seed_node or nd.is_leaf():
            continue
        desc = frozenset(lf.taxon.label for lf in nd.leaf_iter())
        if 1 < len(desc) < n:
            clades.add(desc)
    return leafset, clades

def normalized_rooted_rf(tree1: dendropy.Tree, tree2: dendropy.Tree) -> float:
    l1 = {lf.taxon.label for lf in tree1.leaf_node_iter()}
    l2 = {lf.taxon.label for lf in tree2.leaf_node_iter()}
    leaves = sorted(l1 & l2)
    n = len(leaves)
    if n < 3:
        return np.nan

    tns = dendropy.TaxonNamespace(leaves)
    t1 = dendropy.Tree.get(
        data=tree1.as_string(schema="newick"),
        schema="newick",
        taxon_namespace=tns
    )
    t2 = dendropy.Tree.get(
        data=tree2.as_string(schema="newick"),
        schema="newick",
        taxon_namespace=tns
    )
    t1.is_rooted = True
    t2.is_rooted = True
    t1.retain_taxa_with_labels(leaves)
    t2.retain_taxa_with_labels(leaves)

    _, c1p = rooted_nontrivial_clades(t1)
    _, c2p = rooted_nontrivial_clades(t2)

    rf = len(c1p.symmetric_difference(c2p))
    denom = 2 * (n - 2)
    return float(rf / denom) if denom > 0 else 0.0

def normalized_bsd(tree1: dendropy.Tree, tree2: dendropy.Tree) -> float:
    """
    Normalized Branch Score Distance based on all pairwise leaf-to-leaf distances.

    Let x and y be the vectors of pairwise distances in a fixed order.
      BSD = ||x - y||_2
    Normalized as:
      BSD_norm = ||x - y||_2 / ||max(x, y)||_2
    where max(x, y) is taken element-wise over the pairwise-distance vectors.
    """
    l1 = {lf.taxon.label for lf in tree1.leaf_node_iter()}
    l2 = {lf.taxon.label for lf in tree2.leaf_node_iter()}
    leaves = sorted(l1 & l2)
    if len(leaves) < 2:
        return float("nan")

    tns = dendropy.TaxonNamespace(leaves)
    t1 = dendropy.Tree.get(
        data=tree1.as_string(schema="newick"),
        schema="newick",
        taxon_namespace=tns,
    )
    t2 = dendropy.Tree.get(
        data=tree2.as_string(schema="newick"),
        schema="newick",
        taxon_namespace=tns,
    )
    t1.is_rooted = True
    t2.is_rooted = True
    t1.retain_taxa_with_labels(leaves)
    t2.retain_taxa_with_labels(leaves)

    pdm1 = t1.phylogenetic_distance_matrix()
    pdm2 = t2.phylogenetic_distance_matrix()
    taxon = {tx.label: tx for tx in tns}

    ss_diff = 0.0
    ss_max = 0.0

    for i in range(len(leaves)):
        a = taxon[leaves[i]]
        for j in range(i + 1, len(leaves)):
            b = taxon[leaves[j]]
            d1 = float(pdm1.distance(a, b))
            d2 = float(pdm2.distance(a, b))
            dd = d1 - d2
            ss_diff += dd * dd
            m = d1 if d1 >= d2 else d2
            ss_max += m * m

    num = math.sqrt(ss_diff)
    denom = math.sqrt(ss_max)
    return float(num / denom) if denom > 0 else 0.0


# Heatmap plotting helpers

def _robust_vmax(vals, q=0.98):
    v = np.asarray(vals, dtype=float)
    v = v[np.isfinite(v)]
    return float(np.quantile(v, q)) if v.size else 1.0

def plot_metric_heatmaps(df: pd.DataFrame, value_col: str, cbar_label: str,
                         out_png: str, out_svg: str,
                         groups_for_plot, method_rows, subset_cols, row_labels):

    ncols = 2
    nrows = math.ceil(len(groups_for_plot) / ncols)

    vmax = _robust_vmax(df[value_col].values, q=0.98)
    vmin = 0.0

    fig, axes = plt.subplots(nrows, ncols, figsize=(10.8, 6.0), sharex=True, sharey=False)
    axes = np.array(axes).ravel()

    last_im = None
    for idx, (ax, g) in enumerate(zip(axes, groups_for_plot)):
        dfg = df[df["group"] == g].copy()
        piv = dfg.pivot(index="row_key", columns="subset", values=value_col)
        piv = piv.reindex(method_rows)
        piv = piv.reindex(columns=subset_cols)

        M = piv.values.astype(float)
        last_im = ax.imshow(M, aspect="auto", vmin=vmin, vmax=vmax, cmap=CMAP)

        ax.set_title(g)

        tick_pos = [0, 4, 9, 14, 19] if len(subset_cols) >= 20 else np.linspace(
            0, len(subset_cols) - 1, min(5, len(subset_cols)), dtype=int
        )
        ax.set_xticks(tick_pos)
        ax.set_xticklabels([subset_cols[i] for i in tick_pos])

        ax.set_yticks(np.arange(len(method_rows)))
        is_left_col = (idx % ncols == 0)
        ax.set_yticklabels(row_labels if is_left_col else [])

        ax.set_xticks(np.arange(-.5, len(subset_cols), 1), minor=True)
        ax.set_yticks(np.arange(-.5, len(method_rows), 1), minor=True)
        ax.grid(which="minor", linestyle="-", linewidth=0.3, alpha=0.25)
        ax.tick_params(which="minor", bottom=False, left=False)

    for j in range(len(groups_for_plot), len(axes)):
        axes[j].axis("off")

    fig.tight_layout(rect=[0.0, 0.0, 0.84, 1.0])
    cax = fig.add_axes([0.87, 0.14, 0.018, 0.72])
    cb = fig.colorbar(last_im, cax=cax)
    cb.set_label(cbar_label)

    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_svg, bbox_inches="tight")
    plt.close(fig)

def plot_combined_heatmaps(df: pd.DataFrame,
                           out_png: str, out_svg: str,
                           groups_for_plot, method_rows, subset_cols, row_labels):
    """
    Combined figure:
      - 2 rows x 4 columns
      - Row 1: normalized RF distance
      - Row 2: normalized BSD
      - One colorbar per row on the far right
      - Method labels only on the leftmost column
      - Panel labels (a) and (b)
    """
    
    groups_for_plot = ["Amphibians", "Mammals", "Sharks", "Squamates"]

    vmax_rf  = _robust_vmax(df["rooted_rf"].values, q=0.98)
    vmax_bsd = _robust_vmax(df["bsd_norm"].values,  q=0.98)

    metric_specs = [
        ("rooted_rf", vmax_rf,  "Normalized RF distance"),
        ("bsd_norm",  vmax_bsd, "Normalized BSD"),
    ]

    fig, axes = plt.subplots(
        2, 4,
        figsize=(18.0, 7.4),
        sharex=False, sharey=False
    )
    axes = np.asarray(axes)

    im_top = None
    im_bot = None

    fig.subplots_adjust(
        left=0.12, right=0.90, top=0.93, bottom=0.10,
        wspace=0.08, hspace=0.22
    )

    # Tick positions
    if len(subset_cols) >= 20:
        tick_pos = [0, 4, 9, 14, 19]
    else:
        tick_pos = np.linspace(0, len(subset_cols) - 1, min(5, len(subset_cols)), dtype=int).tolist()

    for r, (value_col, vmax, _label) in enumerate(metric_specs):
        for c, g in enumerate(groups_for_plot):
            ax = axes[r, c]
            dfg = df[df["group"] == g].copy()

            # If group is absent, render an empty panel (NaNs) instead of failing.
            if dfg.empty:
                M = np.full((len(method_rows), len(subset_cols)), np.nan, dtype=float)
            else:
                piv = dfg.pivot(index="row_key", columns="subset", values=value_col)
                piv = piv.reindex(method_rows)
                piv = piv.reindex(columns=subset_cols)
                M = piv.values.astype(float)

            im = ax.imshow(
                M,
                aspect="auto",
                vmin=0.0, vmax=vmax,
                cmap=CMAP,
                interpolation="nearest"
            )

            if r == 0:
                im_top = im
                ax.set_title(g, fontsize=18)
            else:
                im_bot = im

            # X ticks/labels (show on both rows)
            ax.set_xticks(tick_pos)
            ax.set_xticklabels([subset_cols[i] for i in tick_pos], fontsize=10)
            ax.tick_params(axis="x", labelbottom=True)

            # X axis label only on bottom row
            if r == 1:
                ax.set_xlabel("Subset", fontsize=16)

            # Y ticks: only leftmost column gets labels
            ax.set_yticks(np.arange(len(method_rows)))
            if c == 0:
                ax.set_yticklabels(row_labels, fontsize=13, fontweight="normal")
            else:
                ax.set_yticklabels([])

            # Gridlines
            ax.set_xticks(np.arange(-.5, len(subset_cols), 1), minor=True)
            ax.set_yticks(np.arange(-.5, len(method_rows), 1), minor=True)
            ax.grid(which="minor", linestyle="-", linewidth=0.3, alpha=0.25)
            ax.tick_params(which="minor", bottom=False, left=False)

    # Panel labels
    fig.text(0.02, 0.92, "(a)", fontsize=18)
    fig.text(0.02, 0.44, "(b)", fontsize=18)

    # Colorbars per row
    top_bbox = axes[0, -1].get_position()
    bot_bbox = axes[1, -1].get_position()

    cax1 = fig.add_axes([top_bbox.x1 + 0.01, top_bbox.y0, 0.012, top_bbox.height])
    cb1 = fig.colorbar(im_top, cax=cax1)
    cb1.set_label("Normalized RF distance", fontsize=14)

    cax2 = fig.add_axes([bot_bbox.x1 + 0.01, bot_bbox.y0, 0.012, bot_bbox.height])
    cb2 = fig.colorbar(im_bot, cax=cax2)
    cb2.set_label("Normalized BSD", fontsize=14)

    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_svg, bbox_inches="tight")
    plt.close(fig)


# Pipeline utilities

def detect_available_groups():
    available = []
    for group, cfg in group_configs.items():
        base_file = os.path.join(DATA_ROOT, cfg["base_file"])
        group_dir = os.path.join(DATA_ROOT, cfg["folder"])
        if os.path.exists(base_file) and os.path.isdir(group_dir):
            available.append(group)
    return [g for g in group_order if g in available]

def find_supertrees_file(group_dir: str, prefix: str):
    direct = os.path.join(group_dir, f"supertrees_{prefix}.txt")
    if os.path.exists(direct):
        return direct
    hits = glob.glob(os.path.join(group_dir, f"supertrees*{prefix}*.txt"))
    return hits[0] if hits else None

def consensus_outfile(group_dir: str, prefix: str, method_key: str):
    return os.path.join(group_dir, f"consensus_{prefix}_{method_key}.txt")

def ensure_consensus_files_for_group(group: str, cfg: dict, base_newicks: list, method_to_files: dict):
    group_dir = os.path.join(DATA_ROOT, cfg["folder"])
    prefix = cfg["prefix"]

    for method_key in method_order:
        out_path = consensus_outfile(group_dir, prefix, method_key)
        if (not FORCE_RECOMPUTE) and os.path.exists(out_path):
            continue

        lines_out = []
        for subset_idx in range(len(base_newicks)):
            files = method_to_files.get(method_key, [])
            if subset_idx >= len(files):
                lines_out.append("")
                continue

            completed_nwks = load_newick_records(files[subset_idx])
            if not completed_nwks:
                lines_out.append("")
                continue

            ref = dendropy.Tree.get(data=base_newicks[subset_idx], schema="newick")
            ref.is_rooted = True
            ref_leaves = {lf.taxon.label for lf in ref.leaf_node_iter()}

            leaf_inter = set(ref_leaves)
            for s in completed_nwks:
                ttmp = dendropy.Tree.get(data=s, schema="newick")
                leaf_inter &= {lf.taxon.label for lf in ttmp.leaf_node_iter()}
            leaf_inter = sorted(leaf_inter)

            if len(leaf_inter) < 2:
                lines_out.append("")
                continue

            topo = build_majority_consensus_newick(completed_nwks, leaf_inter, min_freq=0.5)
            cons_with_lengths = set_branch_lengths_by_split_average(
                topo, completed_nwks, drop_internal_labels=DROP_INTERNAL_LABELS
            )
            lines_out.append(ensure_semicolon(cons_with_lengths))

        Path(out_path).write_text("\n".join(lines_out) + "\n", encoding="utf-8")
        print(f"[OK] wrote consensus file: {out_path}")


# Main

def main():
    processing_groups = detect_available_groups()
    if not processing_groups:
        raise RuntimeError("No groups detected. Check folder names and DATA_ROOT.")

    if USE_CACHE_IF_AVAILABLE and (not FORCE_RECOMPUTE) and os.path.exists(METRICS_PKL):
        df = pd.read_pickle(METRICS_PKL)
        print("[CACHE] loaded", METRICS_PKL)
    else:
        rows = []

        for group in processing_groups:
            cfg = group_configs[group]
            group_dir = os.path.join(DATA_ROOT, cfg["folder"])
            base_file = os.path.join(DATA_ROOT, cfg["base_file"])
            prefix = cfg["prefix"]

            base_newicks = load_newick_records(base_file)
            if not base_newicks:
                continue

            supertrees_path = find_supertrees_file(group_dir, prefix)
            if supertrees_path is None:
                raise FileNotFoundError(f"Missing supertrees_{prefix}.txt in {group_dir}")

            super_nwks = load_newick_records(supertrees_path)
            num_subsets = min(len(base_newicks), len(super_nwks))

            method_to_files = {}
            for method_key, (suffix, _pretty) in methods.items():
                method_dirname = f"completed_multisets_{prefix}_{suffix}"
                method_dir = os.path.join(group_dir, method_dirname)
                files = sorted(
                    glob.glob(os.path.join(method_dir, "*.txt")),
                    key=extract_subset_index
                )
                method_to_files[method_key] = files

            ensure_consensus_files_for_group(group, cfg, base_newicks[:num_subsets], method_to_files)

            consensus_files = {}
            for method_key in method_order:
                cpath = consensus_outfile(group_dir, prefix, method_key)
                consensus_files[method_key] = load_newick_records(cpath) if os.path.exists(cpath) else []

            for subset_idx in range(num_subsets):
                subset_id = subset_idx + 1

                ref = dendropy.Tree.get(data=base_newicks[subset_idx], schema="newick")
                ref.is_rooted = True

                st = dendropy.Tree.get(data=super_nwks[subset_idx], schema="newick")
                st.is_rooted = True

                rows.append({
                    "group": group,
                    "subset": subset_id,
                    "row_key": "MRPLUS_SUPERTREE",
                    "label": "MR(+) supertree",
                    "rooted_rf": normalized_rooted_rf(ref, st),
                    "bsd_norm": normalized_bsd(ref, st),
                })

                for method_key in method_order:
                    c_lines = consensus_files.get(method_key, [])
                    if subset_idx >= len(c_lines):
                        continue
                    nwk = c_lines[subset_idx].strip()
                    if not nwk:
                        continue

                    cons = dendropy.Tree.get(data=nwk, schema="newick")
                    cons.is_rooted = True

                    rows.append({
                        "group": group,
                        "subset": subset_id,
                        "row_key": method_key,
                        "label": f"{methods[method_key][1]} consensus",
                        "rooted_rf": normalized_rooted_rf(ref, cons),
                        "bsd_norm": normalized_bsd(ref, cons),
                    })

        df = pd.DataFrame(rows)
        df.to_pickle(METRICS_PKL)
        df.to_csv(METRICS_CSV, index=False)
        print("[OK] wrote", METRICS_PKL)
        print("[OK] wrote", METRICS_CSV)

    # Plot controls
    groups_for_plot = ["Amphibians", "Mammals", "Sharks", "Squamates"]
    subset_cols = sorted(df["subset"].unique().tolist())

    # Row order (methods + MR(+) supertree)
    method_rows = method_order + ["MRPLUS_SUPERTREE"]

    display_label = {
        "original":         "Proposed\nconsensus",
        "at_root":          "Root Attach\nconsensus",
        "nearest_parent":   "Nearest Parent\nconsensus",
        "no_mcs":           "No MCS\nconsensus",
        "supertree":        "Supertree-based\nconsensus",
        "MRPLUS_SUPERTREE": "MR(+)\nsupertree",
    }
    row_labels = [display_label[k] for k in method_rows]

    plot_combined_heatmaps(
        df=df,
        out_png=HEAT_COMBINED_PNG,
        out_svg=HEAT_COMBINED_SVG,
        groups_for_plot=groups_for_plot,
        method_rows=method_rows,
        subset_cols=subset_cols,
        row_labels=row_labels,
    )

    print("[DONE] heatmaps saved to", OUTDIR)
    print("[DONE] combined SVG:", HEAT_COMBINED_SVG)

if __name__ == "__main__":
    main()

[CACHE] loaded eval_consensus_vs_supertrees\consensus_vs_supertree_metrics.pkl
[DONE] heatmaps saved to eval_consensus_vs_supertrees
[DONE] combined SVG: eval_consensus_vs_supertrees\heatmap_consensus_vs_supertree_combined.svg
